# Bước 2: Khám phá dữ liệu (EDA)
## Đề 13: Phân tích hội thoại trong DVKH trực tuyến
**Mục tiêu:** Trực quan hóa phân phối dữ liệu, phát hiện pattern

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('../data/processed/cleaned_data.csv')
conv_df = df.drop_duplicates('conv_id')
customer_df = df[df['speaker'] == 'customer']

print(f'Dataset: {len(df)} dòng, {len(conv_df)} đoạn hội thoại')

## 2.1 Phân phối chủ đề

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
topic_counts = conv_df['topic_name'].value_counts().head(20)
topic_counts.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Số đoạn hội thoại')
ax.set_ylabel('Chủ đề')
ax.set_title('Top 20 chủ đề phổ biến nhất')
ax.invert_yaxis()
for i, v in enumerate(topic_counts.values):
    ax.text(v + 1, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../results/figures/01_eda/topic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Tổng số chủ đề: {conv_df["topic_name"].nunique()}')

## 2.2 Phân phối mức hài lòng

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
sat_counts = conv_df['final_satisfaction'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].pie(sat_counts.values, labels=sat_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Phân phối mức hài lòng')

# Bar chart
sat_counts.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_ylabel('Số đoạn')
axes[1].set_title('Số lượng theo mức hài lòng')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(sat_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/01_eda/satisfaction_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 Phân phối mức từ lóng

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
slang_counts = conv_df['slang_level'].value_counts()
slang_colors = ['#3498db', '#e67e22', '#e74c3c']
slang_counts.plot(kind='bar', ax=ax, color=slang_colors, edgecolor='black')
ax.set_ylabel('Số đoạn')
ax.set_title('Phân phối mức từ lóng')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(slang_counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/01_eda/slang_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Phân phối số lượt lời mỗi đoạn

In [ ]:
turns_per_conv = df.groupby('conv_id')['turn'].max()

fig, ax = plt.subplots(figsize=(10, 5))
turns_per_conv.hist(bins=range(1, turns_per_conv.max()+2), ax=ax,
                    color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xlabel('Số lượt lời')
ax.set_ylabel('Số đoạn hội thoại')
ax.set_title('Phân phối số lượt lời mỗi đoạn hội thoại')
ax.axvline(turns_per_conv.mean(), color='red', linestyle='--', label=f'TB: {turns_per_conv.mean():.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/01_eda/turn_length_hist.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Word Cloud

In [ ]:
try:
    from wordcloud import WordCloud

    all_customer_text = ' '.join(customer_df['message_clean'].dropna().astype(str))

    # Stopwords tiếng Việt cơ bản
    stopwords_vi = {'của', 'và', 'là', 'có', 'được', 'cho', 'không', 'này', 'đã',
                    'với', 'các', 'trong', 'những', 'nhưng', 'một', 'người', 'để',
                    'từ', 'hay', 'tôi', 'mình', 'bạn', 'shop', 'ơi', 'ạ', 'nhé',
                    'dạ', 'thì', 'mà', 'cái', 'đó', 'rồi', 'nha', 'luôn', 'quá',
                    'lắm', 'vậy', 'thôi', 'đi', 'the', 'giúp', 'giúm', 'cho',
                    'anh', 'chị', 'em'}

    wc = WordCloud(width=1200, height=600, background_color='white',
                   max_words=100, colormap='viridis',
                   stopwords=stopwords_vi, min_font_size=10)
    wc.generate(all_customer_text)

    fig, ax = plt.subplots(figsize=(15, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud — Tin nhắn khách hàng', fontsize=16)
    plt.tight_layout()
    plt.savefig('../results/figures/01_eda/wordcloud.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print('Cần cài: pip install wordcloud')

## 2.6 Top 20 từ xuất hiện nhiều nhất (khách hàng)

In [ ]:
stopwords_vi = {'của', 'và', 'là', 'có', 'được', 'cho', 'không', 'này', 'đã',
                'với', 'các', 'trong', 'những', 'nhưng', 'một', 'người', 'để',
                'từ', 'hay', 'tôi', 'mình', 'bạn', 'shop', 'ơi', 'ạ', 'nhé',
                'dạ', 'thì', 'mà', 'cái', 'đó', 'rồi', 'nha', 'luôn', 'quá',
                'lắm', 'vậy', 'thôi', 'đi', 'giúp', 'cho', 'anh', 'chị', 'em',
                'bị', 'lại', 'cũng', 'như', 'nào', 'còn', 'khi', 'sao'}

all_words = ' '.join(customer_df['message_clean'].dropna().astype(str)).split()
filtered_words = [w for w in all_words if w not in stopwords_vi and len(w) > 1]
word_freq = Counter(filtered_words).most_common(20)

fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*word_freq)
ax.barh(range(len(words)), counts, color='steelblue', edgecolor='black')
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_xlabel('Tần suất')
ax.set_title('Top 20 từ xuất hiện nhiều nhất (khách hàng)')
plt.tight_layout()
plt.savefig('../results/figures/01_eda/top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Hài lòng theo chủ đề

In [ ]:
top_topics = conv_df['topic_name'].value_counts().head(10).index
subset = conv_df[conv_df['topic_name'].isin(top_topics)]

ct = pd.crosstab(subset['topic_name'], subset['final_satisfaction'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(12, 6))
ct[['satisfied', 'neutral', 'unsatisfied']].plot(kind='barh', stacked=True, ax=ax,
    color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='black')
ax.set_xlabel('Tỉ lệ (%)')
ax.set_title('Mức hài lòng theo chủ đề (Top 10)')
ax.legend(title='Mức hài lòng')
plt.tight_layout()
plt.savefig('../results/figures/01_eda/satisfaction_by_topic.png', dpi=150, bbox_inches='tight')
plt.show()